### Retrieval-Augmented Generation (RAG) Chatbot

A RAG chatbot connects a large language model (LLM) to data sources like PDFs and websites. Doing so allows the chatbot to give context-aware and accurate answers to frequently-asked questions.

### Prepare Data

- Load the UW-Madison DAPIR website, extract text content, and clean text content.
- Chunking: Split documents into smaller, semantically-meaningful chunks (~500-1000 tokens).

In [1]:
# Import libraries
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
from typing import List, Dict
import re

# Configuration
BASE_URL = "https://data.wisc.edu/"
MAX_LINKS = 3  # Start small for testing

In [2]:
def fetch_webpage(url: str) -> BeautifulSoup:
    """
    Fetch a webpage and return a BeautifulSoup object.
    
    Args:
        url: The URL to fetch
        
    Returns:
        BeautifulSoup object containing parsed HTML
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        print(f"✓ Successfully fetched: {url}")
        return soup
        
    except requests.exceptions.RequestException as e:
        print(f"✗ Error fetching {url}: {e}")
        return None


def extract_links(soup: BeautifulSoup, base_url: str) -> List[str]:
    """
    Extract all links from a webpage and convert to absolute URLs.
    
    Args:
        soup: BeautifulSoup object of the page
        base_url: Base URL for converting relative links
        
    Returns:
        List of absolute URLs
    """
    if not soup:
        return []
    
    links = []
    base_domain = urlparse(base_url).netloc
    
    for anchor in soup.find_all('a', href=True):
        href = anchor['href']
        
        # Convert to absolute URL
        absolute_url = urljoin(base_url, href)
        
        # Filter: only include links from the same domain
        if urlparse(absolute_url).netloc == base_domain:
            # Skip anchors, PDFs, and other non-HTML content
            if not any(absolute_url.endswith(ext) for ext in ['.pdf', '.jpg', '.png', '.zip']):
                if '#' not in absolute_url or absolute_url.split('#')[0] not in links:
                    links.append(absolute_url.split('#')[0])  # Remove anchor
    
    # Remove duplicates while preserving order
    seen = set()
    unique_links = []
    for link in links:
        if link not in seen and link != base_url:
            seen.add(link)
            unique_links.append(link)
    
    return unique_links


# Fetch the main page
print("Fetching main webpage...")
main_soup = fetch_webpage(BASE_URL)

# Extract all links from main page
all_links = extract_links(main_soup, BASE_URL)
print(f"\nFound {len(all_links)} unique links on the main page")

# Select first MAX_LINKS for testing
selected_links = all_links[:MAX_LINKS]
print(f"\nSelected {len(selected_links)} links for initial scraping:")
for i, link in enumerate(selected_links, 1):
    print(f"  {i}. {link}")

Fetching main webpage...
✓ Successfully fetched: https://data.wisc.edu/

Found 39 unique links on the main page

Selected 3 links for initial scraping:
  1. https://data.wisc.edu/about-us/
  2. https://data.wisc.edu/dapir-staff/
  3. https://data.wisc.edu/contact-us/


In [3]:
def extract_text_content(soup: BeautifulSoup, url: str) -> Dict[str, str]:
    """
    Extract meaningful text content from a webpage.
    
    Args:
        soup: BeautifulSoup object of the page
        url: Source URL for metadata
        
    Returns:
        Dictionary with extracted content and metadata
    """
    if not soup:
        return None
    
    # Remove script, style, and navigation elements
    for element in soup(['script', 'style', 'nav', 'footer', 'header', 'iframe']):
        element.decompose()
    
    # Extract title
    title = soup.find('title')
    title_text = title.get_text().strip() if title else "No Title"
    
    # Extract main content
    # Try to find main content area (common patterns)
    main_content = (
        soup.find('main') or 
        soup.find('article') or 
        soup.find('div', {'class': ['content', 'main-content', 'page-content']}) or
        soup.find('body')
    )
    
    # Extract text from paragraphs, headings, and lists
    text_elements = []
    
    if main_content:
        for tag in main_content.find_all(['h1', 'h2', 'h3', 'h4', 'p', 'li']):
            text = tag.get_text().strip()
            if text and len(text) > 20:  # Filter out very short snippets
                text_elements.append(text)
    
    combined_text = '\n\n'.join(text_elements)
    
    return {
        'url': url,
        'title': title_text,
        'content': combined_text,
        'word_count': len(combined_text.split())
    }


# Scrape content from selected links
print("Scraping content from selected pages...\n")
scraped_data = []

for i, url in enumerate(selected_links, 1):
    print(f"[{i}/{len(selected_links)}] Scraping: {url}")
    
    # Fetch the page
    soup = fetch_webpage(url)
    
    # Extract content
    page_data = extract_text_content(soup, url)
    
    if page_data and page_data['word_count'] > 0:
        scraped_data.append(page_data)
        print(f"    Title: {page_data['title'][:60]}...")
        print(f"    Words: {page_data['word_count']}")
    else:
        print(f"    ⚠ No content extracted")
    
    # Be respectful - don't hammer the server
    time.sleep(1)
    print()

print(f"✓ Successfully scraped {len(scraped_data)} pages")
print(f"Total words collected: {sum(d['word_count'] for d in scraped_data)}")

Scraping content from selected pages...

[1/3] Scraping: https://data.wisc.edu/about-us/
✓ Successfully fetched: https://data.wisc.edu/about-us/
    Title: About Us – Data, Academic Planning & Institutional Research ...
    Words: 285

[2/3] Scraping: https://data.wisc.edu/dapir-staff/
✓ Successfully fetched: https://data.wisc.edu/dapir-staff/
    Title: Our Staff – Data, Academic Planning & Institutional Research...
    Words: 184

[3/3] Scraping: https://data.wisc.edu/contact-us/
✓ Successfully fetched: https://data.wisc.edu/contact-us/
    Title: Contact us – Data, Academic Planning & Institutional Researc...
    Words: 87

✓ Successfully scraped 3 pages
Total words collected: 556


In [4]:
def clean_text(text: str) -> str:
    """
    Clean and normalize text content.
    
    Args:
        text: Raw text to clean
        
    Returns:
        Cleaned text
    """
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s.,!?;:()\-\'\"]+', '', text)
    
    # Remove repeated punctuation
    text = re.sub(r'([.,!?;:]){2,}', r'\1', text)
    
    return text.strip()


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[Dict[str, any]]:
    """
    Split text into overlapping chunks for better context preservation.
    
    Args:
        text: Text to chunk
        chunk_size: Target size in words
        overlap: Number of overlapping words between chunks
        
    Returns:
        List of text chunks with metadata
    """
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        chunk_text = ' '.join(chunk_words)
        
        if len(chunk_words) > 20:  # Only keep substantial chunks
            chunks.append({
                'text': chunk_text,
                'word_count': len(chunk_words),
                'chunk_index': len(chunks)
            })
    
    return chunks


# Process scraped data: clean and chunk
print("Processing scraped content...\n")
processed_documents = []

for page_data in scraped_data:
    # Clean the content
    cleaned_content = clean_text(page_data['content'])
    
    # Chunk the content
    chunks = chunk_text(cleaned_content, chunk_size=500, overlap=50)
    
    # Add metadata to each chunk
    for chunk in chunks:
        processed_documents.append({
            'source_url': page_data['url'],
            'page_title': page_data['title'],
            'chunk_text': chunk['text'],
            'chunk_index': chunk['chunk_index'],
            'word_count': chunk['word_count']
        })
    
    print(f"✓ Processed: {page_data['title'][:50]}...")
    print(f"  Created {len(chunks)} chunks")

print(f"\n✓ Total documents created: {len(processed_documents)}")
print(f"  Average chunk size: {sum(d['word_count'] for d in processed_documents) / len(processed_documents):.0f} words")

Processing scraped content...

✓ Processed: About Us – Data, Academic Planning & Institutional...
  Created 1 chunks
✓ Processed: Our Staff – Data, Academic Planning & Institutiona...
  Created 1 chunks
✓ Processed: Contact us – Data, Academic Planning & Institution...
  Created 1 chunks

✓ Total documents created: 3
  Average chunk size: 185 words


### Convert and Store Embeddings

- Convert text chunks into vector embeddings with a pre-trained model (HuggingFace sentence-transformers).
- Store embeddings in a vector database (ChromaDB) to index chunks with embeddings and metadata.

In [5]:
# Import libraries
from sentence_transformers import SentenceTransformer
import numpy as np

# Load pre-trained embedding model
print("Loading sentence-transformer model...")
print("This may take a moment on first run as the model downloads (~400MB)\n")

# Using 'all-MiniLM-L6-v2': Fast, efficient, good for general use
# Alternatives: 'all-mpnet-base-v2' (more accurate but slower)
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"✓ Model loaded: {model.get_sentence_embedding_dimension()}-dimensional embeddings")

# Generate embeddings for all chunks
print(f"\nGenerating embeddings for {len(processed_documents)} text chunks...")

# Extract just the text for embedding
chunk_texts = [doc['chunk_text'] for doc in processed_documents]

# Generate embeddings (batched for efficiency)
embeddings = model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✓ Successfully generated {len(embeddings)} embeddings")
print(f"  Embedding shape: {embeddings.shape}")
print(f"  Embedding dimension: {embeddings.shape[1]}")
print(f"  Data type: {embeddings.dtype}")

# Add embeddings to processed documents
for i, doc in enumerate(processed_documents):
    doc['embedding'] = embeddings[i]

# Verify
sample_embedding = processed_documents[0]['embedding']
print(f"\n✓ Embeddings added to documents")
print(f"  Sample embedding preview: [{sample_embedding[:5]}...]")
print(f"  Embedding vector norm: {np.linalg.norm(sample_embedding):.4f}")

Loading sentence-transformer model...
This may take a moment on first run as the model downloads (~400MB)

✓ Model loaded: 384-dimensional embeddings

Generating embeddings for 3 text chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Successfully generated 3 embeddings
  Embedding shape: (3, 384)
  Embedding dimension: 384
  Data type: float32

✓ Embeddings added to documents
  Sample embedding preview: [[-0.02139355 -0.00384603 -0.05842846  0.01708994 -0.02967962]...]
  Embedding vector norm: 1.0000


### Retrieval

- Convert user prompts into embeddings using the same, pre-trained model.
- Execute similarity search to find the most relevant text chunks.

In [6]:
"""
=============================================================================
CHROMADB - VECTOR DATABASE FOR RAG SYSTEMS
=============================================================================

WHAT IS CHROMADB?
-----------------
ChromaDB is an open-source vector database designed for AI applications.
It stores vector embeddings alongside their metadata and enables fast 
similarity search.

WHY USE A VECTOR DATABASE?
--------------------------
Regular databases (SQL, etc.) store exact data and use exact matching.
Vector databases store embeddings and use "similarity search" to find
related content even if the words are different.

Example:
  User asks: "How do I apply to college?"
  Vector DB finds chunks about: "admissions process", "application requirements"
  (even though the exact words differ)

WHAT CHROMADB DOES IN THIS PROJECT:
-----------------------------------
1. STORAGE: Stores our text chunks + their embeddings + metadata (URLs, titles)
2. INDEXING: Organizes embeddings for fast retrieval (HNSW algorithm)
3. SEARCH: When user asks a question:
   - Convert question to embedding (same model)
   - Find the K most similar embeddings in the database
   - Return the corresponding text chunks
4. PERSISTENCE: Saves to disk so we don't re-scrape/re-embed every time

HOW IT WORKS:
------------
- Collections: Like tables in SQL, organize related documents
- Documents: Our text chunks with metadata
- Embeddings: Vector representations that enable similarity search
- Queries: Convert user questions to embeddings, find nearest neighbors

=============================================================================
"""

# Import libraries
import chromadb
from chromadb.config import Settings

print("Setting up ChromaDB vector database...\n")

# Initialize ChromaDB client with persistent storage
# This creates a folder './chroma_db' to save our data
client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(
        anonymized_telemetry=False,  # Disable telemetry
        allow_reset=True
    )
)

# Create or get a collection (like a table in SQL)
# Collections organize related documents
collection_name = "DAPIR_website_chunks"

# Delete existing collection if it exists (for fresh start during development)
try:
    client.delete_collection(name=collection_name)
    print(f"✓ Deleted existing collection: {collection_name}")
except:
    pass

# Create new collection
collection = client.create_collection(
    name=collection_name,
    metadata={
        "description": "DAPIR website content chunks with embeddings",
        "embedding_model": "all-MiniLM-L6-v2",
        "created_date": "2026-02-02"
    }
)

print(f"✓ Created collection: {collection_name}")

# Prepare data for ChromaDB
# ChromaDB expects: ids, documents, embeddings, metadatas

ids = [f"chunk_{i}" for i in range(len(processed_documents))]

documents = [doc['chunk_text'] for doc in processed_documents]

embeddings_list = [doc['embedding'].tolist() for doc in processed_documents]

metadatas = [{
    'source_url': doc['source_url'],
    'page_title': doc['page_title'],
    'chunk_index': doc['chunk_index'],
    'word_count': doc['word_count']
} for doc in processed_documents]

print(f"\nPrepared {len(ids)} documents for storage")

# Add documents to ChromaDB
print("Adding documents to vector database...")

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings_list,
    metadatas=metadatas
)

print(f"✓ Successfully added {len(ids)} documents to ChromaDB")

# Verify storage
print(f"\n{'='*80}")
print("VECTOR DATABASE SUMMARY")
print(f"{'='*80}")
print(f"Collection name: {collection_name}")
print(f"Total documents: {collection.count()}")
print(f"Storage location: ./chroma_db/")
print(f"{'='*80}")

# Test retrieval with a sample query
print("\nTesting vector search with sample query...")
test_query = "What is this website about?"
print(f"Query: '{test_query}'")

# Convert query to embedding using the same model
query_embedding = model.encode([test_query]).tolist()

# Search for top 3 most similar chunks
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print(f"\nTop 3 most relevant chunks:")
for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0], 
    results['metadatas'][0],
    results['distances'][0]
), 1):
    print(f"\n{i}. Similarity score: {1 - distance:.4f}")  # Convert distance to similarity
    print(f"   Source: {metadata['page_title'][:50]}...")
    print(f"   URL: {metadata['source_url']}")
    print(f"   Text preview: {doc[:150]}...")

print(f"\n{'='*80}")
print("✓ ChromaDB setup complete and tested!")
print("✓ Vector database is ready for RAG queries")
print(f"{'='*80}")

Setting up ChromaDB vector database...

✓ Created collection: DAPIR_website_chunks

Prepared 3 documents for storage
Adding documents to vector database...
✓ Successfully added 3 documents to ChromaDB

VECTOR DATABASE SUMMARY
Collection name: DAPIR_website_chunks
Total documents: 3
Storage location: ./chroma_db/

Testing vector search with sample query...
Query: 'What is this website about?'

Top 3 most relevant chunks:

1. Similarity score: -0.6690
   Source: Contact us – Data, Academic Planning & Institution...
   URL: https://data.wisc.edu/contact-us/
   Text preview: There are several ways to connect with DAPIR. Reach out to individual staff. Email infoaccesswisc.edu for questions related to the InfoAccess data war...

2. Similarity score: -0.6971
   Source: About Us – Data, Academic Planning & Institutional...
   URL: https://data.wisc.edu/about-us/
   Text preview: DAPIR manages the structures needed to ensure the availability of accessible, usable, high-quality data. This allows

### Generation

- Pass the retrieved chunks as **context** to an open-source LLM (Llamma).
- LLM prompts = user prompt + retrieved context + sys. instructions
- Generate an answer grounded in the source documents.
- Optionally include references to source documents.

### Model Evaluation

- Test the chatbot with sample FAQs.
- Are the answers accurate and helpful?